# Fog-aware detection training pipeline

Fog/no-fog feature extraction (unsupervised) -> conditional DCP dehazing + exposure enhancement on the foggy subset only -> COCO-to-YOLO conversion -> YOLO training (50 epochs for iteration, 100 for the final model).

**Important:** update `DATA_DIR` in the CONFIG cell below to point at your dataset root — the folder containing `train/`, `valid/`, `test/` subfolders, each with its own images and its own `_annotations.coco.json`. Any README file in those folders is ignored (only the JSON + image files are read).

**Note on "scikit-learn exposure enhancement":** scikit-learn (`sklearn`) has no image-exposure functions — it's a general ML library (the `KMeans`/`StandardScaler` used below *are* from sklearn, for the fog clustering). Exposure enhancement (CLAHE / adaptive histogram equalization) lives in **scikit-image** (`skimage.exposure`), which is what's used in the exposure-enhancement section.

## 0. Install dependencies

In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA GeForce RTX 4050 Laptop GPU


## 1. Config
All paths and the ablation toggles (`APPLY_DCP`, `APPLY_EXPOSURE`) live here. Re-running the notebook with different toggles reproduces the DCP-only / exposure-only / baseline ablation variants discussed earlier.

In [4]:
import os
from pathlib import Path

CONFIG = {
    # --- raw data: dataset root containing train/ valid/ test/ subfolders ---
    "DATA_DIR": "G:\Projects\Low Visibility\low visibility Final.v2i.coco",                       # <-- point this at your dataset root
    "COCO_JSON_NAME": "_annotations.coco.json",    # Roboflow's default COCO export filename
    # maps our internal split name -> the actual subfolder name in DATA_DIR
    "SPLIT_DIRS": {"train": "train", "val": "valid", "test": "test"},

    # --- working dirs ---
    "YOLO_DATASET_DIR": "data/yolo_dataset",       # final images/ + labels/ per-split structure
    "FOG_MODEL_PATH": "fog_model.pkl",

    # --- fog scoring (matches the earlier fog_classifier.py) ---
    "SCORE_RESOLUTION": (256, 256),
    "DARK_CHANNEL_PATCH": 15,

    # --- preprocessing ablation toggles ---
    "APPLY_DCP": True,
    "APPLY_EXPOSURE": True,

    # --- training image size (final resize; must match inference pipeline) ---
    "IMG_SIZE": 640,
    "RESIZE_INTERPOLATION": "INTER_AREA",           # keep identical to the deployed pipeline

    "SEED": 42,

    # --- YOLO training ---
    "YOLO_WEIGHTS": "yolov8n.pt",                  # swap for yolov8s.pt etc. if you want a bigger backbone
    "EPOCHS": 50,                                   # use 50 for ablation/iteration runs; bump to 100 only for the final chosen model
    "BATCH": 16,

    # --- hyperparameter tuning ---
    "TUNE_ENABLED": True,
    "TUNE_ITERATIONS": 50,     # number of candidate hyperparameter sets to try
    "TUNE_EPOCHS": 30,         # short training run per candidate (cheaper than full 100 epochs)
    "TUNE_OPTIMIZER": "AdamW",

    # --- device ---
    "DEVICE": 0,   # GPU index (0 = your RTX 4050); use "cpu" to force CPU
}

Path(CONFIG["YOLO_DATASET_DIR"]).mkdir(parents=True, exist_ok=True)

## 2. Load COCO annotations for each split
Each of `train/`, `valid/`, `test/` has its own COCO JSON. We load all three with `pycocotools`, and build the category-id -> YOLO-index mapping once from the `train` split (Roboflow keeps category ids consistent across splits for the same dataset version; a mismatch check below will warn you if that's not true for your export).

In [5]:
from pycocotools.coco import COCO

coco_by_split = {}
image_dir_by_split = {}
for split, subdir in CONFIG["SPLIT_DIRS"].items():
    split_dir = Path(CONFIG["DATA_DIR"]) / subdir
    json_path = split_dir / CONFIG["COCO_JSON_NAME"]
    coco_by_split[split] = COCO(str(json_path))
    image_dir_by_split[split] = str(split_dir)
    print(f"{split}: {len(coco_by_split[split].getImgIds())} images")

cat_ids = sorted(coco_by_split["train"].getCatIds())
cat_id_to_name = {c["id"]: c["name"] for c in coco_by_split["train"].loadCats(cat_ids)}
cat_id_to_yolo_idx = {cid: i for i, cid in enumerate(cat_ids)}
yolo_idx_to_name = {i: cat_id_to_name[cid] for cid, i in cat_id_to_yolo_idx.items()}
print(f"classes: {list(cat_id_to_name.values())}")

# sanity check: make sure valid/test use the same category id -> name mapping as train
for split in ["val", "test"]:
    other_cats = {c["id"]: c["name"] for c in coco_by_split[split].loadCats(coco_by_split[split].getCatIds())}
    if other_cats != cat_id_to_name:
        print(f"[warn] category mapping in '{split}' differs from 'train': {other_cats} vs {cat_id_to_name}")

loading annotations into memory...
Done (t=0.40s)
creating index...
index created!
train: 7695 images
loading annotations into memory...
Done (t=0.18s)
creating index...
index created!
val: 1847 images
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
test: 1380 images
classes: ['Bike-car-person', 'Bike', 'Car', 'Person']


## 3. Fog / no-fog feature extraction (unsupervised)
Same 3 features as the standalone `fog_classifier.py`: dark-channel mean, true Laplacian variance, mean saturation. Features are computed across **all three splits combined** and clustered together with a single k-means (k=2) fit, so the fog/clear decision boundary is identical regardless of which split an image belongs to.

In [6]:
import cv2
import numpy as np
import pandas as pd
import joblib
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def estimate_fog_features(img_bgr, size=CONFIG["SCORE_RESOLUTION"], patch=CONFIG["DARK_CHANNEL_PATCH"]):
    img = cv2.resize(img_bgr, size, interpolation=cv2.INTER_AREA)
    min_channel = np.min(img, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (patch, patch))
    dark_channel = cv2.erode(min_channel, kernel)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return {
        "mean_dark_intensity": float(np.mean(dark_channel)),
        "laplacian_var": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        "mean_saturation": float(np.mean(hsv[:, :, 1])),
    }

rows = []
for split, coco in coco_by_split.items():
    image_dir = image_dir_by_split[split]
    for img_info in coco.loadImgs(coco.getImgIds()):
        path = str(Path(image_dir) / img_info["file_name"])
        img = cv2.imread(path)
        if img is None:
            print(f"[warn] could not read {path}")
            continue
        feats = estimate_fog_features(img)
        feats["split"] = split
        feats["image_id"] = img_info["id"]
        feats["path"] = path
        feats["file_name"] = img_info["file_name"]
        rows.append(feats)

fog_df = pd.DataFrame(rows)
print(f"Scored {len(fog_df)} images across all splits")

Scored 10922 images across all splits


In [7]:
feature_cols = ["mean_dark_intensity", "laplacian_var", "mean_saturation"]
scaler = StandardScaler()
X = scaler.fit_transform(fog_df[feature_cols])

km = KMeans(n_clusters=2, n_init=10, random_state=CONFIG["SEED"])
fog_df["cluster"] = km.fit_predict(X)

cluster_means = fog_df.groupby("cluster")["mean_dark_intensity"].mean()
fog_cluster = cluster_means.idxmax()  # higher dark-channel mean -> fog
fog_df["fog_label"] = (fog_df["cluster"] == fog_cluster).map({True: "fog", False: "clear"})

print(fog_df.groupby("split")["fog_label"].value_counts())
joblib.dump({"scaler": scaler, "kmeans": km, "fog_cluster": fog_cluster}, CONFIG["FOG_MODEL_PATH"])

split  fog_label
test   fog           948
       clear         432
train  fog          5443
       clear        2252
val    fog          1332
       clear         515
Name: count, dtype: int64


['fog_model.pkl']

## 4. Dark Channel Prior dehazing
Full DCP: dark channel -> atmospheric light estimate -> transmission map -> guided-filter refinement (fast approximation of soft matting) -> scene radiance recovery.

In [8]:
def dark_channel(img, patch=15):
    min_channel = np.min(img, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (patch, patch))
    return cv2.erode(min_channel, kernel)

def estimate_atmospheric_light(img, dark, top_percent=0.001):
    h, w = dark.shape
    n_pixels = max(int(h * w * top_percent), 1)
    flat_dark = dark.reshape(-1)
    flat_img = img.reshape(-1, 3)
    idx = np.argpartition(flat_dark, -n_pixels)[-n_pixels:]
    # among the brightest dark-channel pixels, take the one with highest intensity in the original image
    brightest = idx[np.argmax(flat_img[idx].sum(axis=1))]
    return flat_img[brightest].astype(np.float64)

def guided_filter(guide, src, radius=40, eps=1e-3):
    guide = guide.astype(np.float64)
    src = src.astype(np.float64)
    mean_g = cv2.boxFilter(guide, cv2.CV_64F, (radius, radius))
    mean_s = cv2.boxFilter(src, cv2.CV_64F, (radius, radius))
    mean_gs = cv2.boxFilter(guide * src, cv2.CV_64F, (radius, radius))
    cov_gs = mean_gs - mean_g * mean_s
    mean_gg = cv2.boxFilter(guide * guide, cv2.CV_64F, (radius, radius))
    var_g = mean_gg - mean_g * mean_g
    a = cov_gs / (var_g + eps)
    b = mean_s - a * mean_g
    mean_a = cv2.boxFilter(a, cv2.CV_64F, (radius, radius))
    mean_b = cv2.boxFilter(b, cv2.CV_64F, (radius, radius))
    return mean_a * guide + mean_b

def estimate_transmission(img, A, patch=15, omega=0.95):
    normalized = img.astype(np.float64) / A
    t = 1 - omega * dark_channel(normalized, patch)
    return t

def dehaze(img_bgr, patch=15, omega=0.95, t0=0.1, guide_radius=40, guide_eps=1e-3):
    img = img_bgr.astype(np.float64)
    dark = dark_channel(img, patch)
    A = estimate_atmospheric_light(img, dark)
    t = estimate_transmission(img, A, patch, omega)
    gray_guide = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float64) / 255.0
    t_refined = guided_filter(gray_guide, t, radius=guide_radius, eps=guide_eps)
    t_refined = np.clip(t_refined, t0, 1.0)
    J = np.empty_like(img)
    for c in range(3):
        J[:, :, c] = (img[:, :, c] - A[c]) / t_refined + A[c]
    return np.clip(J, 0, 255).astype(np.uint8)

## 5. Exposure enhancement (scikit-image)
CLAHE via `skimage.exposure.equalize_adapthist`, applied on the L channel in LAB space so color isn't distorted the way per-channel RGB equalization would.

In [9]:
from skimage import exposure, color

def enhance_exposure(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB) / 255.0
    lab = color.rgb2lab(rgb)
    l_channel = lab[:, :, 0] / 100.0  # normalize L to [0, 1] for equalize_adapthist
    l_eq = exposure.equalize_adapthist(l_channel, clip_limit=0.01)
    lab[:, :, 0] = l_eq * 100.0
    rgb_eq = np.clip(color.lab2rgb(lab), 0, 1)
    return cv2.cvtColor((rgb_eq * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

## 6. Build the preprocessed dataset, per split
Respects the train/valid/test split your dataset already came with — no re-shuffling. Branches per image using `fog_label`: fog images get DCP (if `APPLY_DCP`) then exposure enhancement (if `APPLY_EXPOSURE`); clear images pass through untouched. Resize happens **last**, after any pixel-level enhancement, and must use the same interpolation as the deployed pipeline. Images are written directly into the final YOLO `images/<split>/` layout (Roboflow's `valid` maps to YOLO's conventional `val` folder name).

In [ ]:
from tqdm import tqdm

interp = getattr(cv2, CONFIG["RESIZE_INTERPOLATION"])
yolo_root = Path(CONFIG["YOLO_DATASET_DIR"])

fog_label_by_key = {(r.split, r.image_id): r.fog_label for r in fog_df.itertuples()}
preprocessed_paths = {}  # (split, image_id) -> output path

for split, coco in coco_by_split.items():
    out_img_dir = yolo_root / "images" / split
    out_img_dir.mkdir(parents=True, exist_ok=True)
    image_dir = image_dir_by_split[split]

    for img_info in tqdm(coco.loadImgs(coco.getImgIds()), desc=split):
        img_id = img_info["id"]
        src_path = str(Path(image_dir) / img_info["file_name"])
        img = cv2.imread(src_path)
        if img is None:
            continue

        if fog_label_by_key.get((split, img_id)) == "fog":
            if CONFIG["APPLY_DCP"]:
                img = dehaze(img)
            if CONFIG["APPLY_EXPOSURE"]:
                img = enhance_exposure(img)
        # clear images (or fog images with both toggles off) pass through untouched

        img = cv2.resize(img, (CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]), interpolation=interp)

        out_path = out_img_dir / img_info["file_name"]
        out_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_path), img)
        preprocessed_paths[(split, img_id)] = str(out_path)

## 7. Convert COCO annotations to YOLO format, per split
Boxes are unaffected by DCP/exposure (pixel-level only), but resizing above changes them — so YOLO's normalized `cx cy w h` format (relative to the *resized* image dimensions) is used, which sidesteps the issue: normalized coords are the same regardless of resolution, as long as the box was computed against the same size the image was actually resized to.

In [ ]:
def coco_bbox_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    return cx, cy, w / img_w, h / img_h

for split, coco in coco_by_split.items():
    out_label_dir = yolo_root / "labels" / split
    out_label_dir.mkdir(parents=True, exist_ok=True)

    for img_info in coco.loadImgs(coco.getImgIds()):
        img_id = img_info["id"]
        if (split, img_id) not in preprocessed_paths:
            continue
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        lines = []
        for ann in anns:
            cx, cy, w, h = coco_bbox_to_yolo(ann["bbox"], img_info["width"], img_info["height"])
            yolo_idx = cat_id_to_yolo_idx[ann["category_id"]]
            lines.append(f"{yolo_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        label_path = out_label_dir / (Path(img_info["file_name"]).stem + ".txt")
        label_path.write_text("\n".join(lines))

    print(f"{split}: wrote {len(list(out_label_dir.glob('*.txt')))} label files")

## 8. Write `data.yaml`

In [ ]:
import yaml

names = [yolo_idx_to_name[i] for i in range(len(yolo_idx_to_name))]
data_yaml = {
    "path": str(yolo_root.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: n for i, n in enumerate(names)},
}

data_yaml_path = yolo_root / "data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(data_yaml_path.read_text())

## 9. Hyperparameter tuning (evolutionary search)
Ultralytics' built-in tuner (`model.tune`) runs `TUNE_ITERATIONS` short trials, each training for `TUNE_EPOCHS` epochs with a mutated set of hyperparameters (learning rate, momentum, weight decay, warmup, loss gains, and augmentation strength — HSV jitter, rotation, translate, scale, mosaic, mixup, etc.), keeping whichever set produced the best fitness (a weighted combination of mAP metrics). This is far cheaper than grid-searching at the full training budget, and the winning hyperparameters are then reused for the real training run in the next section.

Set `CONFIG["TUNE_ENABLED"] = False` to skip this and train section 10 with Ultralytics' defaults instead.

In [ ]:
from ultralytics import YOLO

best_hyp_path = None
if CONFIG["TUNE_ENABLED"]:
    tune_model = YOLO(CONFIG["YOLO_WEIGHTS"])
    tune_model.tune(
        data=str(data_yaml_path),
        epochs=CONFIG["TUNE_EPOCHS"],
        iterations=CONFIG["TUNE_ITERATIONS"],
        optimizer=CONFIG["TUNE_OPTIMIZER"],
        imgsz=CONFIG["IMG_SIZE"],
        batch=CONFIG["BATCH"],
        seed=CONFIG["SEED"],
        device=CONFIG["DEVICE"],
        plots=False,
        save=False,
        val=True,
    )
    best_hyp_path = Path("runs") / "detect" / "tune" / "best_hyperparameters.yaml"
    if best_hyp_path.exists():
        print(best_hyp_path.read_text())
    else:
        print("[warn] expected tuning output not found at", best_hyp_path)
        best_hyp_path = None

## 10. Train YOLO (50 epochs for iteration runs — set `EPOCHS` to 100 for the final model)

In [ ]:
model = YOLO(CONFIG["YOLO_WEIGHTS"])

train_kwargs = dict(
    data=str(data_yaml_path),
    epochs=CONFIG["EPOCHS"],
    imgsz=CONFIG["IMG_SIZE"],
    batch=CONFIG["BATCH"],
    seed=CONFIG["SEED"],
    device=CONFIG["DEVICE"],
    name=f"dcp{int(CONFIG['APPLY_DCP'])}_exp{int(CONFIG['APPLY_EXPOSURE'])}",
)
if best_hyp_path is not None:
    train_kwargs["cfg"] = str(best_hyp_path)  # applies the tuned hyperparameters as overrides

results = model.train(**train_kwargs)

## 11. Ablation runs
To reproduce the baseline / DCP-only / exposure-only variants alongside this combined run: change `APPLY_DCP` / `APPLY_EXPOSURE` in the CONFIG cell, then re-run from **section 6 onward** (fog labels and DCP/exposure functions don't need to be redone — only the preprocessed-image generation, label conversion, and training do). Each run's `name=` in the training call keeps its outputs separated under `runs/detect/`.